# Model Context Protocol (MCP)

Model Context Protocol (MCP) is an open protocol that standardizes how applications provide tools and context to LLMs. LangChain agents can use tools defined on MCP servers using the langchain-mcp-adapters library.

pip install langchain-mcp-adapters

<b>langchain-mcp-adapters</b> enables agents to use tools defined across one or more MCP servers.

MultiServerMCPClient is <b>stateless by default</b>. 

1. Each tool invocation creates a fresh MCP ClientSession
2. executes the tool
3. then cleans up. 

See the stateful sessions section for more details.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient  ## Import here
from langchain.agents import create_agent


client = MultiServerMCPClient(  ## define
    {
        "math": {
            "transport": "stdio",  # Local subprocess communication
            "command": "python",
            # Absolute path to your math_server.py file
            "args": ["/path/to/math_server.py"],
        },
        "weather": {
            "transport": "http",  # HTTP-based remote server
            # Ensure you start your weather server on port 8000
            "url": "http://localhost:8000/mcp",
        }
    }
)

tools = await client.get_tools()  ## this is not common syntax as normally await need to be in async func
# but it can use due to langchain special RECL something
agent = create_agent( 
    "claude-sonnet-4-5-20250929",
    tools 
)
## ainvoke (asynchronous invoke) not just invoke 
math_response = await agent.ainvoke( 
    {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]}
)
weather_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what is the weather in nyc?"}]}
)